# 04 — Modeling

**Project:** Forecasting the Effects of Exchange Rate Fluctuations on US Trade Flows  
**Phase:** CRISP-DM 4 — Modeling  
**Author:** Francisco Giordano Rigon

---

## Structure

- **24 target series:** 3 countries × 2 directions (exports/imports) × 4 sectors
- **3 algorithms:** ARIMA, Random Forest, LightGBM
- **72 models total**
- **Train:** Jan 2010 – Dec 2021 (144 months)
- **Test:** Jan 2022 – Dec 2024 (36 months)

## Notebook sections

1. Setup
2. Load data
3. Define targets and features
4. ARIMA models
5. Random Forest models
6. LightGBM models
7. Results comparison

---
## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pickle
import json
import warnings
from pathlib import Path

# Stats / ML
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import itertools

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
np.random.seed(42)

# --- Paths ---
PROC    = Path('../data/processed')
MODELS  = Path('../models')
RESULTS = Path('../results')

# --- Train / test split ---
TRAIN_END  = '2021-12-01'
TEST_START = '2022-01-01'

# --- Country pairs ---
PARTNERS = {
    'CAN': 'Canada',
    'MEX': 'Mexico',
    'BRA': 'Brazil',
}

# --- 24 target series ---
# Raw column names in the processed datasets
TARGETS = [
    'exports_total',
    'exports_commodities',
    'exports_manufactured_goods',
    'exports_high-tech',
    'imports_total',
    'imports_commodities',
    'imports_manufactured_goods',
    'imports_high-tech',
]

# --- Feature columns (73 inputs) ---
# Defined after loading data (Section 3)

print('Setup complete.')
print(f'Train: up to {TRAIN_END[:7]} | Test: from {TEST_START[:7]}')
print(f'Countries: {list(PARTNERS.keys())}')
print(f'Targets per country: {len(TARGETS)}')
print(f'Total models: {len(PARTNERS) * len(TARGETS) * 3} (3 algorithms)')

Setup complete.
Train: up to 2021-12 | Test: from 2022-01
Countries: ['CAN', 'MEX', 'BRA']
Targets per country: 8
Total models: 72 (3 algorithms)


---
## 2. Load Data

In [2]:
datasets = {}

for iso in PARTNERS:
    df = pd.read_csv(PROC / ('dataset_' + iso.lower() + '.csv'), index_col=0, parse_dates=True)
    datasets[iso] = df

print('=== Datasets loaded ===')
for iso, df in datasets.items():
    nan_total = df.isna().sum().sum()
    start = df.index[0].strftime('%Y-%m')
    end   = df.index[-1].strftime('%Y-%m')
    target_cols = [c for c in df.columns
                   if ('exports' in c or 'imports' in c)
                   and not c.startswith('log_')
                   and not c.startswith('diff_')]
    nulls_in_targets = df[target_cols].isna().sum().sum()
    print()
    print(iso + ' (' + PARTNERS[iso] + ')')
    print('  Shape:      ' + str(df.shape))
    print('  Date range: ' + start + ' to ' + end)
    print('  NaN total:  ' + str(nan_total) + '  (expected: lag/diff leading rows only)')
    print('  Targets (' + str(len(target_cols)) + '): ' + str(target_cols))
    if nulls_in_targets > 0:
        print('  WARNING: NaN in targets!')
    else:
        print('  Targets: no missing values')


=== Datasets loaded ===

CAN (Canada)
  Shape:      (180, 97)
  Date range: 2010-01 to 2024-12
  NaN total:  166  (expected: lag/diff leading rows only)
  Targets (8): ['exports_total', 'exports_commodities', 'exports_manufactured_goods', 'exports_high-tech', 'imports_total', 'imports_commodities', 'imports_manufactured_goods', 'imports_high-tech']
  Targets: no missing values

MEX (Mexico)
  Shape:      (180, 97)
  Date range: 2010-01 to 2024-12
  NaN total:  166  (expected: lag/diff leading rows only)
  Targets (8): ['exports_total', 'exports_commodities', 'exports_manufactured_goods', 'exports_high-tech', 'imports_total', 'imports_commodities', 'imports_manufactured_goods', 'imports_high-tech']
  Targets: no missing values

BRA (Brazil)
  Shape:      (180, 97)
  Date range: 2010-01 to 2024-12
  NaN total:  166  (expected: lag/diff leading rows only)
  Targets (8): ['exports_total', 'exports_commodities', 'exports_manufactured_goods', 'exports_high-tech', 'imports_total', 'imports_co

---
## 3. Define Features and Targets

This section:
- Identifies the 73 feature columns (X) used by RF and LightGBM
- Defines helper functions to prepare train/test splits for each algorithm
- Defines the metrics function used to evaluate all 72 models

In [3]:
# --- Feature columns: everything that is NOT a trade flow target ---
# Exclude: exports_*, imports_*, log_exports_*, log_imports_*,
#          diff_log_exports_*, diff_log_imports_*
def get_feature_cols(df: pd.DataFrame) -> list[str]:
    """Return the 73 feature column names (X matrix)."""
    exclude = [c for c in df.columns
               if 'exports' in c or 'imports' in c]
    return [c for c in df.columns if c not in exclude]

# Verify on CAN dataset
feature_cols = get_feature_cols(datasets['CAN'])
print('Feature columns (X):', len(feature_cols))
print()

# Group them for readability
groups = {
    'Exchange rate (bilateral)': [c for c in feature_cols if c.startswith('FX_') and 'lag' not in c and 'ma' not in c and 'pct' not in c],
    'REER':                      [c for c in feature_cols if c.startswith('REER_') and 'lag' not in c and 'pct' not in c],
    'US macro':                  ['FEDFUNDS','CPI_USA','GDP_USA','UNRATE_USA','INDPRO_USA'],
    'Partner rates':             [c for c in feature_cols if c.startswith('RATE_')],
    'Partner CPI':               [c for c in feature_cols if c.startswith('CPI_') and c != 'CPI_USA'],
    'Partner GDP':               [c for c in feature_cols if c.startswith('GDP_') and c != 'GDP_USA'],
    'Partner IndPro':            [c for c in feature_cols if c.startswith('INDPRO_') and c != 'INDPRO_USA'],
    'Commodities':               ['WTI_oil','Soybean','Iron_ore'],
    'Crisis dummies':            ['dummy_gfc','dummy_covid'],
    'Lags':                      [c for c in feature_cols if 'lag' in c],
    'Moving averages':           [c for c in feature_cols if '_ma' in c],
    'Pct changes':               [c for c in feature_cols if '_pct' in c],
    'Calendar':                  ['month','quarter','year'],
}

for group, cols in groups.items():
    valid = [c for c in cols if c in feature_cols]
    print(f'  {group}: {len(valid)} columns')


Feature columns (X): 73

  Exchange rate (bilateral): 3 columns
  REER: 4 columns
  US macro: 5 columns
  Partner rates: 3 columns
  Partner CPI: 3 columns
  Partner GDP: 3 columns
  Partner IndPro: 3 columns
  Commodities: 3 columns
  Crisis dummies: 2 columns
  Lags: 28 columns
  Moving averages: 9 columns
  Pct changes: 4 columns
  Calendar: 3 columns


In [4]:
# --- Helper functions ---

def get_train_test_ml(df: pd.DataFrame, target: str) -> tuple:
    """
    Prepare X and y for RF and LightGBM.

    The target is the LOG of the raw trade flow (log1p).
    Features are the 73 columns, with NaN rows dropped.

    Returns: X_train, X_test, y_train, y_test (all DataFrames/Series)
    """
    feature_cols = get_feature_cols(df)
    log_target = 'log_' + target

    # Drop rows where any feature or target is NaN
    # (this removes the first 12 rows where lag12 is NaN)
    cols_needed = feature_cols + [log_target]
    clean = df[cols_needed].dropna()

    X = clean[feature_cols]
    y = clean[log_target]

    X_train = X[X.index <= TRAIN_END]
    X_test  = X[X.index >= TEST_START]
    y_train = y[y.index <= TRAIN_END]
    y_test  = y[y.index >= TEST_START]

    return X_train, X_test, y_train, y_test


def get_train_test_arima(df: pd.DataFrame, target: str) -> tuple:
    """
    Prepare the series for ARIMA.

    ARIMA uses the DIFFERENCED log series (diff_log_target),
    which is stationary. Returns train and test Series.
    """
    diff_target = 'diff_log_' + target

    series = df[diff_target].dropna()

    train = series[series.index <= TRAIN_END]
    test  = series[series.index >= TEST_START]

    return train, test


def mape(y_true: pd.Series, y_pred: np.ndarray) -> float:
    """Mean Absolute Percentage Error (%). Skips zero true values."""
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def compute_metrics(y_true: pd.Series, y_pred: np.ndarray,
                    iso: str, target: str, algorithm: str) -> dict:
    """
    Compute MAE, RMSE, MAPE for one model.
    All metrics are on the LOG scale (same scale as the model output).
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_val = mape(y_true, pd.Series(y_pred, index=y_true.index))

    return {
        'algorithm': algorithm,
        'country':   iso,
        'target':    target,
        'MAE':       round(mae, 6),
        'RMSE':      round(rmse, 6),
        'MAPE':      round(mape_val, 4),
    }


# --- Verify helpers on one example ---
X_train, X_test, y_train, y_test = get_train_test_ml(datasets['CAN'], 'exports_total')
train_s, test_s = get_train_test_arima(datasets['CAN'], 'exports_total')

print('ML split — CAN exports_total (log scale):')
print('  X_train:', X_train.shape, '| y_train:', y_train.shape)
print('  X_test: ', X_test.shape,  '| y_test: ', y_test.shape)
print()
print('ARIMA split — CAN exports_total (diff_log):')
print('  train:', train_s.shape, '| test:', test_s.shape)
print('  train mean:', round(train_s.mean(), 4), '| train std:', round(train_s.std(), 4))


ML split — CAN exports_total (log scale):
  X_train: (132, 73) | y_train: (132,)
  X_test:  (36, 73) | y_test:  (36,)

ARIMA split — CAN exports_total (diff_log):
  train: (143,) | test: (36,)
  train mean: 0.0033 | train std: 0.084
